# Ranking Competition 

Метод: дообучение jinaai/jina-reranker-v2-base-multilingual на размеченных парах query–passage с BCEWithLogitsLoss; после обучения – исправление поврежденной кодировки только в тестовых текстах и инференс

Система исходного эксперимента: Linux (Kaggle), NVIDIA Tesla T4, Python 3.13.16, PyTorch 2.11.0+cu130, Transformers 5.20.2.

## Параметры и окружение

In [2]:
import os, sys, platform, math, random, re
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_ID = "jinaai/jina-reranker-v2-base-multilingual"
MAX_LENGTH = 768
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUM = 8
EPOCHS = 1
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

random.seed(67)
np.random.seed(67)
torch.manual_seed(67)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("OS:", platform.system(), platform.release())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if USE_AMP else "нет")
print("AMP:", USE_AMP)


OS: Linux 6.18.55+
Python: 3.13.16
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Device: cuda:0
GPU: Tesla T4
AMP: True


## Поиск данных и загрузка

In [3]:
def locate_data():
    if os.environ.get("AMML_DATA_DIR"):
        root = Path(os.environ["AMML_DATA_DIR"]).expanduser()
        if not all((root / x).is_file() for x in ("train.csv", "test.csv", "sample_submission.csv")):
            raise FileNotFoundError(f"В AMML_DATA_DIR отсутствуют файлы соревнования: {root}")
        return root.resolve()
    roots = [Path.cwd() / "data", Path.cwd(),
             Path("/kaggle/input/datasets/alionabir/domashka"),
             Path("/kaggle/input/amml-ranking-competition-2026")]
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        roots += sorted({p.parent for p in kaggle.rglob("train.csv")})
    matches = [p for p in roots if all((p / x).is_file() for x in ("train.csv", "test.csv", "sample_submission.csv"))]
    if not matches:
        raise FileNotFoundError("Не найдены train.csv, test.csv и sample_submission.csv. Задайте AMML_DATA_DIR или положите их в ./data")
    return matches[0].resolve()

DATA_DIR = locate_data()
OUTPUT_DIR = Path("/kaggle/working") / "outputs" if Path("/kaggle/working").exists() else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_all = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str, "query_id": str})
test = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str, "query_id": str})
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})
assert {"id", "query_id", "query", "title", "text", "relevance"}.issubset(train_all.columns)
assert {"id", "query_id", "query", "title", "text"}.issubset(test.columns)
assert {"id", "prediction"}.issubset(sample.columns)
assert train_all.id.is_unique and test.id.is_unique and sample.id.is_unique
assert set(sample.id) == set(test.id)
assert set(train_all.query_id).isdisjoint(set(test.query_id))
assert train_all.relevance.isin([0, 1]).all()
for frame in (train_all, test):
    for col in ("query", "title", "text"):
        frame[col] = frame[col].fillna("").astype(str)
print("DATA_DIR:", DATA_DIR)
print("Train:", len(train_all), "rows;", train_all.query_id.nunique(), "queries")
print("Test:", len(test), "rows;", test.query_id.nunique(), "queries")


DATA_DIR: /kaggle/input/datasets/alionabir/domashka
Train: 31530 rows; 3042 queries
Test: 13507 rows; 1305 queries


## Разделение по запросам 

In [4]:
from sklearn.model_selection import GroupShuffleSplit

In [7]:
split = GroupShuffleSplit(n_splits=1, test_size=0.20,random_state=33)

idx_train, idx_valid = next(split.split(train_all, groups=train_all["query_id"]))

fit = train_all.iloc[idx_train].reset_index(drop=True)
valid = train_all.iloc[idx_valid].reset_index(drop=True)

assert set(fit["query_id"]).isdisjoint(set(valid["query_id"]))

print("Train:", fit.shape)
print("Validation:", valid.shape)
print("Train queries:", fit["query_id"].nunique())
print("Validation queries:", valid["query_id"].nunique())

Train: (25188, 12)
Validation: (6342, 12)
Train queries: 2433
Validation queries: 609


## Загрузка токенизатора и модели

In [8]:
import transformers.models.xlm_roberta.modeling_xlm_roberta as xlm_module

In [9]:
if not hasattr(xlm_module, "create_position_ids_from_input_ids"):
    def create_position_ids_from_input_ids(input_ids, padding_idx, past_key_values_length=0):
        mask = input_ids.ne(padding_idx).int()
        incremental_indices = (torch.cumsum(mask, dim=1).type_as(mask) + past_key_values_length) * mask
        return incremental_indices.long() + padding_idx
    xlm_module.create_position_ids_from_input_ids = create_position_ids_from_input_ids

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, trust_remote_code=True, num_labels=1,
    ignore_mismatched_sizes=True, torch_dtype="float32",
).to(device=DEVICE, dtype=torch.float32)
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

configuration_xlm_roberta.py:   0%|          | 0.00/2.73k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual:
- configuration_xlm_roberta.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (14 outputs left out)

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.


Loading weights:   0%|          | 0/153 [00:00<?, ?it/s]

[transformers] You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.
[transformers] XLMRobertaForSequenceClassification does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


Model loaded: jinaai/jina-reranker-v2-base-multilingual


## Формирование пар и DataLoader

In [10]:
def make_passage(title, text):
    title, text = str(title).strip(), str(text).strip()
    return f"Title: {title}\nPassage: {text}" if title else f"Passage: {text}"

class PairDataset(Dataset):
    def __init__(self, frame, labeled=True):
        self.queries = frame["query"].tolist()
        self.passages = [make_passage(t, x) for t, x in zip(frame["title"], frame["text"])]
        self.labels = frame["relevance"].to_numpy(np.float32) if labeled else None
    def __len__(self):
        return len(self.queries)
    def __getitem__(self, idx):
        label = 0.0 if self.labels is None else float(self.labels[idx])
        return self.queries[idx], self.passages[idx], label

def collate(batch):
    queries, passages, labels = zip(*batch)
    encoded = tokenizer(list(queries), list(passages), padding=True,
                        truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded["labels"] = torch.tensor(labels, dtype=torch.float32)
    return encoded

train_loader = DataLoader(PairDataset(fit), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=0)
valid_loader = DataLoader(PairDataset(valid), batch_size=EVAL_BATCH_SIZE, shuffle=False,
                          collate_fn=collate, num_workers=0)
print("Train batches:", len(train_loader))


Train batches: 12594


##  Дообучение реранкера (BCE)

In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, round(WARMUP_RATIO * updates_per_epoch * EPOCHS)),
    num_training_steps=updates_per_epoch * EPOCHS,)
loss_fn = torch.nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def get_logits(outputs):
    logits = outputs.logits
    return logits.squeeze(-1) if logits.ndim == 2 else logits

model.train()
optimizer.zero_grad(set_to_none=True)
for epoch in range(EPOCHS):
    running_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        labels = batch.pop("labels").to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        with (torch.autocast("cuda", dtype=torch.float16) if USE_AMP else nullcontext()):
            logits = get_logits(model(**inputs))
            loss = loss_fn(logits.float(), labels)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: epoch={epoch}, batch={step}")
        remaining_in_group = len(train_loader) - ((step - 1) // GRAD_ACCUM) * GRAD_ACCUM
        divisor = min(GRAD_ACCUM, remaining_in_group)
        scaler.scale(loss / divisor).backward()
        running_loss += float(loss.item())
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        if step % 500 == 0 or step == len(train_loader):
            print(f"Epoch {epoch+1}/{EPOCHS}, batch {step}/{len(train_loader)}, BCE={running_loss/step:.5f}", flush=True)
print("Training finished")

CHECKPOINT = OUTPUT_DIR / "jina_finetuned"
CHECKPOINT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(CHECKPOINT)
tokenizer.save_pretrained(CHECKPOINT)
print("Checkpoint:", CHECKPOINT)


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch 1/1, batch 500/12594, BCE=0.39596
Epoch 1/1, batch 1000/12594, BCE=0.35506
Epoch 1/1, batch 1500/12594, BCE=0.34239
Epoch 1/1, batch 2000/12594, BCE=0.33129
Epoch 1/1, batch 2500/12594, BCE=0.32403
Epoch 1/1, batch 3000/12594, BCE=0.32412
Epoch 1/1, batch 3500/12594, BCE=0.32295
Epoch 1/1, batch 4000/12594, BCE=0.31620
Epoch 1/1, batch 4500/12594, BCE=0.31211
Epoch 1/1, batch 5000/12594, BCE=0.30762
Epoch 1/1, batch 5500/12594, BCE=0.30542
Epoch 1/1, batch 6000/12594, BCE=0.30331
Epoch 1/1, batch 6500/12594, BCE=0.30113
Epoch 1/1, batch 7000/12594, BCE=0.29709
Epoch 1/1, batch 7500/12594, BCE=0.29491
Epoch 1/1, batch 8000/12594, BCE=0.29352
Epoch 1/1, batch 8500/12594, BCE=0.29368
Epoch 1/1, batch 9000/12594, BCE=0.29244
Epoch 1/1, batch 9500/12594, BCE=0.29250
Epoch 1/1, batch 10000/12594, BCE=0.29114
Epoch 1/1, batch 10500/12594, BCE=0.29042
Epoch 1/1, batch 11000/12594, BCE=0.28813
Epoch 1/1, batch 11500/12594, BCE=0.28760
Epoch 1/1, batch 12000/12594, BCE=0.28634
Epoch 1/1, b

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint: /kaggle/working/outputs/jina_finetuned


## Валидация

In [12]:
@torch.inference_mode()
def predict(loader, label):
    model.eval()
    scores = []
    for i, batch in enumerate(loader, 1):
        batch.pop("labels", None)
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        with (torch.autocast("cuda", dtype=torch.float16) if USE_AMP else nullcontext()):
            logits = get_logits(model(**inputs))
        scores.extend(logits.float().cpu().tolist())
        if i % 250 == 0 or i == len(loader):
            print(f"{label}: {i}/{len(loader)}", flush=True)
    arr = np.asarray(scores, dtype=np.float64)
    assert np.isfinite(arr).all()
    return arr

def ndcg_at_10(frame):
    values = []
    for _, group in frame.groupby("query_id", sort=False):
        group = group.sort_values(["prediction", "id"], ascending=[False, True], kind="mergesort")
        y = group["relevance"].to_numpy(dtype=np.float64)
        k = min(10, len(y))
        discount = 1 / np.log2(np.arange(2, k+2))
        dcg = float((y[:k] * discount).sum())
        ideal = np.sort(y)[::-1][:k]
        idcg = float((ideal * discount).sum())
        values.append(dcg / idcg if idcg else 0.0)
    return float(np.mean(values))

valid_pred = valid[["id", "query_id", "relevance"]].copy()
valid_pred["prediction"] = predict(valid_loader, "Validation")
print(f"Validation NDCG@10: {ndcg_at_10(valid_pred):.6f}")
valid_pred.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)


Validation: 250/1586
Validation: 500/1586
Validation: 750/1586
Validation: 1000/1586
Validation: 1250/1586
Validation: 1500/1586
Validation: 1586/1586
Validation NDCG@10: 0.959570


##  Исправление кодировки после обучения – только тест

In [13]:
def repair_mojibake(value):
    if not isinstance(value, str) or not value:
        return value
    if not re.search(r'[РС][Ѐ-ӿ]|[РС][\u0080-\u00BF]', value):
        return value
    try:
        repaired = value.encode("cp1251").decode("utf-8")
    except (UnicodeError, ValueError):
        return value
    def badness(s):
        return len(re.findall(r'Р[°-я]|С[°-я]', s))
    return repaired if badness(repaired) < badness(value) else value
test_fixed = test.copy(deep=True)
for col in ("query", "title", "text"):
    test_fixed[col] = test_fixed[col].map(repair_mojibake)
    print(f"{col}: исправлено {(test_fixed[col] != test[col]).sum()} строк")


query: исправлено 0 строк
title: исправлено 0 строк
text: исправлено 72 строк


## Финальный инференс и сабмишен

In [15]:
test_loader_fixed = DataLoader(PairDataset(test_fixed, labeled=False),
                               batch_size=EVAL_BATCH_SIZE, shuffle=False,
                               collate_fn=collate, num_workers=0)
test_scores = test_fixed[["id"]].copy()
test_scores["prediction"] = predict(test_loader_fixed, label ='testim')
submission = sample[["id"]].merge(test_scores, on="id", how="left", validate="one_to_one")
assert len(submission) == len(test) == len(sample)
assert submission.id.equals(sample.id)
assert np.isfinite(submission.prediction.to_numpy()).all()
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved:", SUBMISSION_PATH.resolve())
print("Rows:", len(submission))
print(submission.head().to_string(index=False))


loba: 250/3377
loba: 500/3377
loba: 750/3377
loba: 1000/3377
loba: 1250/3377
loba: 1500/3377
loba: 1750/3377
loba: 2000/3377
loba: 2250/3377
loba: 2500/3377
loba: 2750/3377
loba: 3000/3377
loba: 3250/3377
loba: 3377/3377
Saved: /kaggle/working/outputs/submission.csv
Rows: 13507
                    id  prediction
r_00100bd94cf8513c6500    2.992188
r_0011d1a81cda1b1c720f   -2.425781
r_00135550b108ebd88432   -3.630859
r_001cac3cb24c86e57ef0   -4.445312
r_001e8de6d4886f85b86e   -1.878906
